In [25]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/prathamd69/datasets/refs/heads/main/healthdataset.csv')

In [30]:
df['daily_calorie_index'] = df['Daily_Calorie_Consumed'] - df['Daily_Calorie_Requirement']
df = df.drop(columns=['Daily_Calorie_Consumed', 'Daily_Calorie_Requirement'])

In [31]:
df = df.drop(columns=['Person_ID'])

In [35]:
from sklearn.preprocessing import OrdinalEncoder
ordered_categories = [['Sedentary', 'Lightly Active', 'Moderately Active', 'Very Active', 'Athlete']]

# Initialize the encoder with your specific hierarchy
encoder = OrdinalEncoder(categories=ordered_categories)
df['Activity_Encoded'] = encoder.fit_transform(df[['Activity_Level']])
df = df.drop(columns=['Activity_Level'])

In [36]:
df = df.drop(columns=['Height_cm','Weight_kg'])

In [38]:
rem = [[ 'Underweight','Healthy', 'Overweight', 'Obese']]

# Initialize the encoder with your specific hierarchy
encoder = OrdinalEncoder(categories=rem)
df['target'] = encoder.fit_transform(df[['Health_Status']])
df = df.drop(columns=['Health_Status'])

In [40]:
df = df.drop(columns=['Diet_Type'])

In [41]:
corr_matrix = df.corr()
corr_matrix.style.background_gradient(
    cmap="coolwarm", vmin=-1, vmax=1
).format(precision=2)

,Age,Gender,BMI,Protein_Intake_g,Carbohydrate_Intake_g,Fat_Intake_g,Water_Intake_Liters,daily_calorie_index,Activity_Encoded,target
Age,1.00,-0.01,0.11,-0.08,-0.10,-0.09,-0.02,0.05,-0.02,0.11
Gender,-0.01,1.00,0.11,0.25,0.31,0.26,0.03,0.18,-0.03,0.11
BMI,0.11,0.11,1.00,0.22,0.27,0.25,-0.12,0.77,-0.18,0.92
Protein_Intake_g,-0.08,0.25,0.22,1.00,-0.13,0.25,0.25,0.30,0.23,0.21
Carbohydrate_Intake_g,-0.10,0.31,0.27,-0.13,1.00,-0.30,0.29,0.38,0.27,0.26
Fat_Intake_g,-0.09,0.26,0.25,0.25,-0.30,1.00,0.28,0.34,0.26,0.25
Water_Intake_Liters,-0.02,0.03,-0.12,0.25,0.29,0.28,1.00,0.02,0.94,-0.14
daily_calorie_index,0.05,0.18,0.77,0.30,0.38,0.34,0.02,1.00,-0.05,0.83
Activity_Encoded,-0.02,-0.03,-0.18,0.23,0.27,0.26,0.94,-0.05,1.00,-0.21
target,0.11,0.11,0.92,0.21,0.26,0.25,-0.14,0.83,-0.21,1.00


In [42]:
df = df.drop(columns=['Gender', 'Water_Intake_Liters'])

In [44]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_data,test_data = train_test_split(df, test_size=0.2, random_state=24)

In [45]:
cols_to_exclude = ['target', 'Activity_Encoded', 'Gender']

exclude_train = [col for col in cols_to_exclude if col in train_data.columns]
exclude_test = [col for col in cols_to_exclude if col in test_data.columns]

train_features_to_scale = train_data.drop(columns=exclude_train)
test_features_to_scale = test_data.drop(columns=exclude_test)
train_scaled_feats = pd.DataFrame(
    scaler.fit_transform(train_features_to_scale),
    columns=train_features_to_scale.columns,
    index=train_data.index
)

test_scaled_feats = pd.DataFrame(
    scaler.transform(test_features_to_scale),
    columns=test_features_to_scale.columns,
    index=test_data.index
)

train_final = pd.concat(
    [train_scaled_feats, train_data[exclude_train]],
    axis=1
)

test_final = pd.concat(
    [test_scaled_feats, test_data[exclude_test]],
    axis=1
)

In [46]:
train_final

,Age,BMI,Protein_Intake_g,Carbohydrate_Intake_g,Fat_Intake_g,daily_calorie_index,target,Activity_Encoded
89,-0.993693,-1.779577,0.712772,-0.619888,-0.595241,-1.424253,1.0,1.0
345,-0.993693,-0.050166,-0.203764,0.664622,-0.811736,0.621606,2.0,1.0
4715,1.493792,0.550324,-0.814787,0.142179,-0.703489,-0.161753,2.0,0.0
2577,0.996295,-1.635460,-0.679535,-0.698132,-0.708095,-1.420451,1.0,2.0
2424,-0.772583,-0.434480,-0.654075,1.011831,0.056548,-1.724668,1.0,3.0
...,...,...,...,...,...,...,...,...
2193,0.609353,-1.010950,-0.988229,-0.490296,-0.512328,-1.256934,1.0,2.0
3473,1.604347,-0.002127,-0.996185,-0.692427,-0.786402,-0.355691,2.0,0.0
343,-1.435912,-0.122225,0.300649,1.376156,-0.247467,0.473300,2.0,2.0
899,1.327960,1.270912,2.165544,-0.526158,-0.021759,1.473413,3.0,0.0


In [76]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Features and target
X = train_final.drop(columns=['target','BMI'])
y = train_final['target']

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Model
lr = LogisticRegression(
    max_iter=300,
    random_state=42
)

# Train
lr.fit(X_train, y_train)

# Predict
y_pred = lr.predict(X_val)

# Metrics
print("Accuracy :", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred, average='weighted'))
print("Recall   :", recall_score(y_val, y_pred, average='weighted'))
print("F1 Score :", f1_score(y_val, y_pred, average='weighted'))

print("\nClassification Report:")
print(classification_report(y_val, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

Accuracy : 0.8208333333333333
Precision: 0.8288555082241998
Recall   : 0.8208333333333333
F1 Score : 0.818768687548512

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      0.90      0.95        42
         1.0       0.87      0.88      0.88       541
         2.0       0.75      0.86      0.80       596
         3.0       0.88      0.61      0.72       261

    accuracy                           0.82      1440
   macro avg       0.88      0.81      0.84      1440
weighted avg       0.83      0.82      0.82      1440


Confusion Matrix:
[[ 38   4   0   0]
 [  0 474  67   0]
 [  0  64 511  21]
 [  0   0 102 159]]


In [63]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Split features and target
X_test = test_final.drop(columns=['target'])
y_test = test_final['target']

# Predict
y_test_pred = lr.predict(X_test)

# Metrics
print("Accuracy :", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred, average='weighted'))
print("Recall   :", recall_score(y_test, y_test_pred, average='weighted'))
print("F1 Score :", f1_score(y_test, y_test_pred, average='weighted'))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Accuracy : 0.9891666666666666
Precision: 0.9892143385070213
Recall   : 0.9891666666666666
F1 Score : 0.989124492758879

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      0.91      0.95        34
         1.0       0.98      0.99      0.99       447
         2.0       0.99      0.99      0.99       513
         3.0       1.00      0.99      0.99       206

    accuracy                           0.99      1200
   macro avg       0.99      0.97      0.98      1200
weighted avg       0.99      0.99      0.99      1200


Confusion Matrix:
[[ 31   3   0   0]
 [  0 444   3   0]
 [  0   4 508   1]
 [  0   0   2 204]]


In [66]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Train
dt = DecisionTreeClassifier(
    random_state=42,
    max_depth=2
)

dt.fit(X_train, y_train)

# Predict
y_pred = dt.predict(X_val)

# Metrics
print("Accuracy :", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred, average='weighted'))
print("Recall   :", recall_score(y_val, y_pred, average='weighted'))
print("F1 Score :", f1_score(y_val, y_pred, average='weighted'))

print("\nClassification Report:")
print(classification_report(y_val, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00        42
         1.0       1.00      1.00      1.00       541
         2.0       1.00      1.00      1.00       596
         3.0       1.00      1.00      1.00       261

    accuracy                           1.00      1440
   macro avg       1.00      1.00      1.00      1440
weighted avg       1.00      1.00      1.00      1440


Confusion Matrix:
[[ 42   0   0   0]
 [  0 541   0   0]
 [  0   0 596   0]
 [  0   0   0 261]]


In [70]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt.feature_importances_
})

print(
    importance.sort_values(
        'Importance',
        ascending=False
    ).head(20)
)

                 Feature  Importance
1                    BMI         1.0
0                    Age         0.0
2       Protein_Intake_g         0.0
3  Carbohydrate_Intake_g         0.0
4           Fat_Intake_g         0.0
5    daily_calorie_index         0.0
6       Activity_Encoded         0.0
